# SMS Spam Detection using Naive Bayes

**Repositori**: Machine Learning  
**Topik**: Implementasi Naive Bayes pada Klasifikasi SMS Spam  
**Dataset**: `SMSSpam.csv`  

---

## Pendahuluan
Proyek ini mendemonstrasikan implementasi **Multinomial Naive Bayes** untuk tugas klasifikasi teks (Spam vs Ham). Naive Bayes sangat cocok untuk klasifikasi teks karena bekerja dengan baik pada data sparse hasil TF-IDF dan memberikan output probabilitas secara alami.

### Alur Kerja (Pipeline):
1. **Data Acquisition**: Mengambil data dari CSV.
2. **EDA (Exploratory Data Analysis)**: Menganalisis distribusi dan karakteristik data.
3. **Data Preparation**: Cleaning dan Label Encoding (Ham=0, Spam=1).
4. **Feature Engineering**: Transformasi teks menggunakan TF-IDF.
5. **Modeling**: Training menggunakan Multinomial Naive Bayes.
6. **Evaluation**: Menggunakan Confusion Matrix, Accuracy, Precision, dan Recall.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('../../').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from utils.evaluation import plot_confusion_matrix, plot_roc_curve, print_classification_report
from utils.preprocessing import train_val_test_split

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Acquisition & Understanding
Memuat dataset dan melihat struktur data awal.

In [ ]:
df = pd.read_csv('../../data/SMSSpam.csv', names=['Label', 'Message'], encoding='latin-1')

print("Shape Dataset:", df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)
Menganalisis karakteristik dataset sebelum preprocessing.

In [ ]:
print("Missing Values:\n", df.isnull().sum())

print("\nDistribusi Label:")
print(df['Label'].value_counts())
print(f"\nPersentase:\n{df['Label'].value_counts(normalize=True).mul(100).round(2)}")

df['Message_Length'] = df['Message'].apply(len)
print(f"\nStatistik Panjang Pesan:")
print(df['Message_Length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='Message_Length', hue='Label', bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Panjang Pesan')
sns.boxplot(data=df, x='Label', y='Message_Length', palette='viridis', ax=axes[1])
axes[1].set_title('Boxplot Panjang Pesan per Label')
plt.tight_layout()
plt.show()

## 3. Data Preparation & Cleaning
Melakukan encoding pada label dan membersihkan data jika diperlukan.

In [ ]:
df['Label_Num'] = df['Label'].map({'ham': 0, 'spam': 1})

# Gunakan stratify karena dataset imbalance (87% ham, 13% spam)
y = df['Label_Num']
X_text_train, X_text_temp, y_train, y_temp = train_test_split(
    df['Message'], y, test_size=0.4, random_state=42, stratify=y
)
X_text_val, X_text_test, y_val, y_test = train_test_split(
    X_text_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Data Training: {len(X_text_train)} sampel")
print(f"Data Validation: {len(X_text_val)} sampel")
print(f"Data Testing: {len(X_text_test)} sampel")

sns.countplot(x='Label', data=df, hue='Label', palette='viridis', legend=False)
plt.title('Distribusi Ham vs Spam')
plt.show()

## 4. Feature Engineering (TF-IDF)
Mengubah teks pesan menjadi representasi angka menggunakan TF-IDF Vectorizer.

In [ ]:
# Fit TF-IDZ HANYA pada training set untuk menghindari data leakage
tfidf = TfidfVectorizer(stop_words='english', max_features=3000, random_state=42)
X_train = tfidf.fit_transform(X_text_train)
X_val = tfidf.transform(X_text_val)
X_test = tfidf.transform(X_text_test)

## 5. Model Training (Multinomial Naive Bayes)
Melatih model dan mengevaluasi performa pada validation set.

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

# Prediksi probabilitas pada validation set untuk ROC Curve
y_val_proba = model.predict_proba(X_val)[:, 1]

# ROC Curve dan optimal threshold pada validation set
roc_auc, optimal_threshold, optimal_fpr, optimal_tpr = plot_roc_curve(
    y_val, y_val_proba,
    title='ROC Curve - Validation Set (Naive Bayes)',
    return_thresholds=True
)

print(f"Threshold Optimal: {optimal_threshold:.3f}")
print(f"TPR (Recall): {optimal_tpr:.3f}")
print(f"FPR: {optimal_fpr:.3f}")

# Prediksi pada test set dengan threshold optimal
y_test_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_test_proba >= optimal_threshold).astype(int)

## 6. Performance Evaluation
Mengevaluasi model menggunakan Confusion Matrix, Accuracy, Precision, Recall, dan F1-Score.

In [ ]:
cm = plot_confusion_matrix(
    y_test, y_pred,
    labels=['Ham', 'Spam'],
    title='Confusion Matrix - Naive Bayes'
)

print("--- Laporan Klasifikasi ---")
print_classification_report(y_test, y_pred, target_names=['Ham', 'Spam'])

# ROC Curve pada test set
plot_roc_curve(
    y_test, y_test_proba,
    title='ROC Curve - Test Set (Naive Bayes)'
)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f}")

## 7. Perbandingan dengan Linear Regression

| Metrik | Linear Regression | Naive Bayes (Setelah Fix) |
|--------|:-:|:-:|
| Accuracy | 0.88 | ? (jalankan sel untuk melihat) |
| Precision (Spam) | 0.55 | ? |
| Recall (Spam) | 0.75 | ? |
| F1-Score (Spam) | 0.63 | ? |
| AUC | ? | ? |

Catatan: Nilai Linear Regression diambil dari notebook `Linear-Regression-SMS.ipynb`. Jalankan semua sel untuk melihat hasil Naive Bayes dan bandingkan!

## Kesimpulan
Multinomial Naive Bayes umumnya lebih unggul daripada Linear Regression untuk klasifikasi teks karena:
1. Naive Bayes memodelkan probabilitas secara alami, memudahkan optimasi threshold.
2. Cocok untuk data sparse hasil TF-IDF.
3. Lebih efisien dan interpretable untuk teks classification.

**Catatan:** Data leakage pada TF-IDF telah diperbaiki. Sekarang `TfidfVectorizer` di-fit hanya pada training set, sehingga evaluasi pada test set valid.

Jalankan semua sel untuk melihat hasilnya dan bandingkan dengan Linear Regression!